# data 

## load

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [ ]:

from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_csv


In [ ]:
representetive_gene_list = (repository_root() / "examples/starmap/representative_genes.txt").read_text().splitlines()
# representetive_gene_list = ["Flt1", "Mgp", "Bgn", "Mylk", "Aqp4", "Rorb", "Cxcl14", "Pcdhgc3", "Ctgf", "Mog", "Enpp2", "Tpbg", "Tcerg1l",
# "Reln", "Pnoc", "Gad1", "Gad2", "Ndnf", "Vip", "Synpr", "Cux2", "Nos1", "Npy", "Lhx6", "Rbp4", "Pcdhgc4",
# "Sema3e", "Sema3c", "Sst", "Syt6", "Sla", "Pcp4", "Foxp2"]  # cell type

In [ ]:
config = load_config("configs/config_zeroshot_starmap.yaml")

# in case you want to change the data name, replicate, folder path, output path
config.data_name = "BZ5"
# IMPORTANT: check replicate"
config.replicate = "_testniche1" 

config.refresh_paths()



In [ ]:
data_path = str(dataset_dir("starmap", config.data_name))

# Load data using the new function
adata = load_spatial_data_csv(
    data_path=data_path,
    main_data_file="data.csv",
    celltype_file="celltype.csv", 
    pos_file="pos.csv",
    domain_file="domain.csv",
    config=config,
    index_col=0,
    first_column_names=True
)

In [ ]:

# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)

# prompt

In [ ]:
# %%
domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6", 10 : "WM"}
# domain_mapping = {2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}
# domain_mapping = {1 : "Layer 1",  3 : "Layer 5", 4 : "Layer 6"}
# domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3",  4 : "Layer 6"}
# domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5"}

config.domain_mapping = domain_mapping

# optional
cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}
config.cell_names_mapping = cell_names_mapping

In [ ]:

# IMPORTANT: check input and prompt_func
if config.Graph_type == "countPlusGenes":
    input_df = neighbor_normalized_df
    df_extra = neighbor_normalized_df_genes
    prompt_func = prompt.zeroshot_celltype_geneorder
elif config.Graph_type == "count":
    input_df = neighbor_normalized_df
    df_extra = None
    prompt_func = prompt.zeroshot_celltype
elif config.Graph_type == "GeneOnly":
    input_df = neighbor_normalized_df_genes
    df_extra = None
    prompt_func = prompt.zeroshot_geneorder
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")


In [ ]:
# print an example
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == 1][3:4]
print(prompt_func(input_df, x, config))



# gpt

## generate json

In [ ]:
# choose correct data and prompt
generate_json_end2end(input_df, 
                      config, 
                      prompt_func, 
                      batch_size = 5000,
                      max_completion_tokens = 512,  # key to control the cost, expecially for o3-mini
                      n_rows = 1,
                      df_extra = df_extra)

## submit

In [ ]:
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_zeroshot_starmap.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_zeroshot{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['zeroshot_gpt4o_mini']


## check results

In [ ]:
gpt_results_df.zeroshot_gpt4o_mini.value_counts()

## save

In [ ]:

adata.obs = adata.obs.join(gpt_results_df)
adata.obs['zeroshot_gpt4o_mini'] = adata.obs['zeroshot_gpt4o_mini'].fillna("unknown")
# refine
adata.obs['zeroshot_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gpt4o_mini'])

adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

print(f"saving to ./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



# Gemini


In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=2000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.zeroshot_celltype, n_rows=1, 
                                                column_name="zeroshot_gemini")



In [ ]:
print(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}_{config.confident_output}{config.replicate}.csv")

In [ ]:
with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}_{config.confident_output}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}_{config.confident_output}{config.replicate}.csv")


In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}_{config.confident_output}{config.replicate}.csv", index_col=0)
gemini_results_df.index = gemini_results_df.index.astype(str)

In [ ]:
gemini_results_df.index.difference(adata.obs.index)

In [ ]:
gemini_results_df.zeroshot_gemini.value_counts()



# run DS

In [ ]:
# def run_deepseek(client, df, config, prompt_func, n_rows=1, start_idx=0, end_idx=None, df_extra=None, 
#                  logprobs=False, max_tokens=1024, column_name=None):
#     """
#     Run DeepSeek API for microenvironment prediction on spatial transcriptomics data.
    
#     Args:
#         client: OpenAI client
#         df: DataFrame containing neighbor counts or spatial data
#         config: Configuration object with model settings
#         prompt_func: Function to generate prompts for each cell
#         n_rows: Number of rows to process per API call (default: 1)
#         start_idx: Starting index for processing (default: 0)
#         end_idx: Ending index for processing (default: None - process all)
#         df_extra: Optional extra DataFrame (e.g., for gene expression data)
#         column_name: Name for output column (default: None)
#         logprobs: Whether to return logprobs (default: False)
#         max_tokens: Maximum number of tokens to generate (default: 1024)
#     Returns:
#         tuple: (deepseek_results_df, store_responses)
#             - deepseek_results_df: DataFrame with predictions
#             - store_responses: List of raw API responses
#     """
    
    
#     df = df.copy()
#     deepseek_results_df = pd.DataFrame()
#     store_responses = []
    
#     if logprobs:
#         top_logprobs = 5
#     else:
#         top_logprobs = None
        
#     if end_idx is None:
#         end_idx = len(df)
    
#     for i in range(start_idx, end_idx, n_rows):
#         if i%10 == 0:
#             print(f"Processed {i} rows")
            
#         # Get row indices for this batch
#         rows = [j for j in range(len(df))[i:i+n_rows]]
        
#         # Generate prompt for this batch
#         if df_extra is None:
#             prompt_q = config.oneshot_prompt + prompt_func(df, rows, config)
#         else:
#             prompt_q = config.oneshot_prompt + prompt_func(df, df_extra, rows, config)
        
#         # Prepare messages for DeepSeek API
#         messages = [
#             {"role": "system", "content": config.system_prompt},
#             {"role": "user", "content": prompt_q}
#         ]
#         try:
#             # Make API call to DeepSeek
#             response = client.chat.completions.create(
#                 model="deepseek-chat",  # DeepSeek's chat model
#                 messages=messages,
#                 stream=False,
#                 max_tokens=max_tokens,
#                 logprobs=logprobs,
#                 top_logprobs=top_logprobs,
#                 response_format={'type': 'json_object'}
#             )
            
#             # Extract response content
#             content = response.choices[0].message.content
#             store_responses.append(content)
            
#             # Process response based on confidence output setting
#             if config.confident_output:
#                 extract_dict = extract_microenvironments_and_confidence(content)
#                 deepseek_results_df = pd.concat([deepseek_results_df, 
#                                                pd.DataFrame({
#                                                    column_name: extract_dict.environments,
#                                                    'confidence': extract_dict.confidence
#                                                }, index=df.index[rows])], ignore_index=False)
#             else:
#                 extract_dict = extract_json_microenvironment(content)
#                 deepseek_results_df = pd.concat([deepseek_results_df, 
#                                                pd.DataFrame(extract_dict.values(), index=df.index[rows])], 
#                                                ignore_index=False)
        
#         except Exception as e:
#             print(f"Error processing rows {rows}: {e}")
#             # Add empty results for failed rows
#             if config.confident_output:
#                 deepseek_results_df = pd.concat([deepseek_results_df, 
#                                                pd.DataFrame({
#                                                    column_name: "Error",
#                                                    'confidence': 0.0
#                                                }, index=rows)], ignore_index=False)
#             else:
#                 deepseek_results_df = pd.concat([deepseek_results_df, 
#                                                pd.DataFrame({"content": "Error"}, index=rows)], 
#                                                ignore_index=False)
#             store_responses.append(f"Error: {e}")
#         # Add delay to respect rate limits
#         time.sleep(0.1)
        
#         # Save intermediate results every 100 rows
#         if (i + 1) % 100 == 0:
#             print(f"Processed {i+1} rows")
            
#             # Create output directory if it doesn't exist
#             output_dir = f'./deepseek_results/'
#             if not os.path.exists(output_dir):
#                 os.makedirs(output_dir)
            
#             # Save responses and results
#             with open(f'{output_dir}{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
#                 pickle.dump(store_responses, file)
#             deepseek_results_df.to_csv(f"{output_dir}{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
    
#     # Set column name if specified
#     if column_name is not None:
#         if len(deepseek_results_df.columns) == 1:
#             deepseek_results_df.columns = [column_name]
    
#     return deepseek_results_df, store_responses


In [ ]:
from openai import OpenAI
import os
import time
import pickle

# Initialize DeepSeek client
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)
results_df, responses = run_deepseek(
    client=client,
    df=input_df,
    config=config,
    prompt_func=prompt_func,  
    start_idx=0,
    end_idx=None,
    df_extra=df_extra,
    logprobs=False,
    max_tokens=1024,
    column_name='zeroshot_deepseek'
)

In [ ]:
results_df.head()

# plot

In [ ]:
#sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gemini", title =  f"zeroshot_gemini")
sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gpt4o_mini", title =  f"zeroshot_gpt4o_mini")
#print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gemini']))
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini']))


In [ ]:

#adata.obs['zeroshot_gemini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gemini'])


In [ ]:
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined']))



In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="zeroshot_gpt4o_mini_refined", title =  f"zeroshot_gpt4o_mini_refined")


In [ ]:
adata.obs['zeroshot_deepseek_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_deepseek'])

sc.pl.scatter(adata, x="x", y="y", color="zeroshot_deepseek_refined", title =  f"zeroshot_deepseek_refined")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_deepseek_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_deepseek_refined']))

In [ ]:
adata.obs.to_csv(f"./deepseek_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

print(f"saving to ./deepseek_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


## plot confident score


In [ ]:
# Import necessary libraries
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Create a new column indicating whether prediction is correct
adata.obs['correct_prediction'] = (adata.obs['zeroshot_gemini'] == adata.obs['niche_name'])

# Group confidence scores by correct/incorrect predictions
correct_conf = adata.obs.loc[adata.obs['correct_prediction'], 'confidence']
incorrect_conf = adata.obs.loc[~adata.obs['correct_prediction'], 'confidence']

# Perform statistical test (Mann-Whitney U test)
stat, pvalue = stats.mannwhitneyu(correct_conf, incorrect_conf)

# Print results
print(f"Mann-Whitney U test for confidence scores between correct and incorrect predictions:")
print(f"Statistic: {stat:.4f}, p-value: {pvalue:.6f}")
print(f"\nMean confidence for correct predictions: {correct_conf.mean():.4f}")
print(f"Mean confidence for incorrect predictions: {incorrect_conf.mean():.4f}")

# Visualize the distributions
plt.figure(figsize=(10, 6))
sns.violinplot(x='correct_prediction', y='confidence', data=adata.obs, inner='quartile')
plt.title('Confidence Score Distribution by Prediction Correctness')
plt.xlabel('Correct Prediction')
plt.ylabel('Confidence Score')
plt.xticks([0, 1], ['Incorrect', 'Correct'])
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add statistical test result to the plot
plt.text(0.5, 0.95, f'Mann-Whitney U p-value: {pvalue:.6f}', 
         horizontalalignment='center', transform=plt.gca().transAxes)

plt.tight_layout()
plt.show()


In [ ]:
gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

In [ ]:
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gemini']))

# Llama 70B Q4

In [ ]:
# from ollama import chat
# from ollama import ChatResponse




In [ ]:
# response: ChatResponse = chat(
#     model='llama70Q4', 
#     messages=[
#         {
#             'role': 'user',
#             'content': prompt.zeroshot_celltype(neighbor_normalized_df, x, config),
#         },
#     ],
#     stream=False,
#     format={
#         "type": "object",
#         "properties": {
#             "Outputs": {
#                 "type": "string"
#             }
#         },
#         "required": [
#             "Outputs"
#         ]
#     },
#     options={
#         "temperature": 0.6
#     }
# )
# niche = json.loads(response['message']['content'])['Outputs']

In [ ]:
# from llama_cpp import Llama
# llm = Llama(model_path=os.environ["LLM_ST_LOCAL_MODEL"], chat_format="chatml")


In [ ]:
# response = llm.create_chat_completion(
#     messages=[
#         {
#             "role": "system",
#             "content": "You are a helpful assistant that outputs in JSON.",
#         },
#         {"role": "user", "content": prompt.zeroshot_celltype(neighbor_normalized_df, x, config)},
#     ],
#     response_format={
#         "type": "json_object",
#         "schema": {
#             "type": "object",
#             "properties": {"Outputs": {"type": "string"}},
#             "required": ["Outputs"],
#         },
#     },
#     temperature=0.7,
# )
# niche = json.loads(response['choices'][0]['message']['content'])['Outputs']

In [ ]:
llama_results_df = pd.read_csv(f"./llama_results/BZ5_zeroshot_llama.csv", index_col=0)
llama_results_df.columns = ['zeroshot_llama']
llama_results_df.index = llama_results_df.index.astype(str)
adata.obs = adata.obs.join(llama_results_df)    



In [ ]:
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_llama']))